In [93]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

In [94]:
df = pd.read_csv(r"C:\Users\2003n\OneDrive - Cal Poly\Desktop\playground-series-s6e4\train.csv")
display(df.head())
display(df.info())

print("Target counts:")
display(df["Irrigation_Need"].value_counts())

print("Target percentages:")
display(df["Irrigation_Need"].value_counts(normalize=True))

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Soil_Type                630000 non-null  object 
 2   Soil_pH                  630000 non-null  float64
 3   Soil_Moisture            630000 non-null  float64
 4   Organic_Carbon           630000 non-null  float64
 5   Electrical_Conductivity  630000 non-null  float64
 6   Temperature_C            630000 non-null  float64
 7   Humidity                 630000 non-null  float64
 8   Rainfall_mm              630000 non-null  float64
 9   Sunlight_Hours           630000 non-null  float64
 10  Wind_Speed_kmh           630000 non-null  float64
 11  Crop_Type                630000 non-null  object 
 12  Crop_Growth_Stage        630000 non-null  object 
 13  Season                   630000 non-null  object 
 14  Irri

None

Target counts:


Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

Target percentages:


Irrigation_Need
Low       0.587170
Medium    0.379483
High      0.033348
Name: proportion, dtype: float64

In [95]:
target = "Irrigation_Need"

X = df.drop(columns=[target])
y = df[target]

# Drop ID column if it exists
if "id" in X.columns:
    X = X.drop(columns=["id"])

display(X.head())

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns

print("Numeric features:")
print(list(numeric_features))

print("\nCategorical features:")
print(list(categorical_features))

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South


Numeric features:
['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']

Categorical features:
['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


In [96]:
from sklearn.preprocessing import PowerTransformer
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("power", PowerTransformer(method="yeo-johnson"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

nb_model_power = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GaussianNB())
])

In [97]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro",
    "recall_macro": "recall_macro"
}

cv_results = cross_validate(
    nb_model,
    X,
    y,
    cv=skf,
    scoring=scoring,
    return_train_score=False
)

cv_summary = pd.DataFrame(cv_results).drop(columns=["fit_time", "score_time"])

display(cv_summary)

cv_results_power = cross_validate(
    nb_model_power,
    X,
    y,
    cv=skf,
    scoring=scoring,
    return_train_score=False
)

pd.DataFrame(cv_results_power).drop(columns=["fit_time", "score_time"]).agg(["mean", "std"]).T

,test_accuracy,test_balanced_accuracy,test_f1_macro,test_recall_macro
0,0.784627,0.780566,0.731621,0.780566
1,0.782921,0.777032,0.728818,0.777032
2,0.783310,0.776871,0.729825,0.776871
3,0.782040,0.776576,0.729025,0.776576
4,0.783444,0.778505,0.730283,0.778505


,mean,std
test_accuracy,0.786981,0.000980
test_balanced_accuracy,0.780219,0.001707
test_f1_macro,0.734185,0.001260
test_recall_macro,0.780219,0.001707


In [98]:
param_grid = {
    "model__var_smoothing": np.logspace(-12, -6, 20)
}

grid_nb = GridSearchCV(
    estimator=nb_model,
    param_grid=param_grid,
    cv=skf,
    scoring="f1_macro",
    n_jobs=-1
)

grid_nb.fit(X, y)

print("Best var_smoothing:", grid_nb.best_params_)
print("Best CV F1 Macro:", grid_nb.best_score_)

best_nb_model = grid_nb.best_estimator_

Best var_smoothing: {'model__var_smoothing': np.float64(2.3357214690901212e-07)}
Best CV F1 Macro: 0.8312904920010331


In [99]:
cv_summary.agg(["mean", "std"]).T

,mean,std
test_accuracy,0.783268,0.000937
test_balanced_accuracy,0.777910,0.001661
test_f1_macro,0.729915,0.001123
test_recall_macro,0.777910,0.001661


In [100]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [101]:
best_nb_model.fit(X_train, y_train)

y_proba = best_nb_model.predict_proba(X_test)

classes = best_nb_model.named_steps["model"].classes_

print("Class order:")
print(classes)

proba_df = pd.DataFrame(
    y_proba,
    columns=classes
)

display(proba_df.head())

Class order:
['High' 'Low' 'Medium']


,High,Low,Medium
0,1.231036e-14,0.988890,0.011110
1,2.878106e-03,0.095663,0.901459
2,9.309487e-07,0.214593,0.785406
3,2.645095e-08,0.972262,0.027738
4,8.290492e-10,0.885457,0.114543


In [102]:
baseline_preds = best_nb_model.predict(X_test)

print("Baseline Classification Report:")
print(classification_report(y_test, baseline_preds))

Baseline Classification Report:
              precision    recall  f1-score   support

        High       0.84      0.67      0.75      4202
         Low       0.92      0.89      0.91     73983
      Medium       0.82      0.87      0.84     47815

    accuracy                           0.88    126000
   macro avg       0.86      0.81      0.83    126000
weighted avg       0.88      0.88      0.88    126000



In [103]:
baseline_cm = confusion_matrix(y_test, baseline_preds, labels=classes)

baseline_cm_df = pd.DataFrame(
    baseline_cm,
    index=[f"Actual {c}" for c in classes],
    columns=[f"Predicted {c}" for c in classes]
)

display(baseline_cm_df)

,Predicted High,Predicted Low,Predicted Medium
Actual High,2823,22,1357
Actual Low,0,66076,7907
Actual Medium,542,5644,41629


In [104]:
focus_class = "High"

high_index = list(classes).index(focus_class)

baseline_recall_high = recall_score(
    y_test,
    baseline_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

baseline_f1_high = f1_score(
    y_test,
    baseline_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

baseline_balanced_acc = balanced_accuracy_score(y_test, baseline_preds)

print("Baseline High Recall:", baseline_recall_high)
print("Baseline High F1:", baseline_f1_high)
print("Baseline Balanced Accuracy:", baseline_balanced_acc)

Baseline High Recall: 0.6718229414564493
Baseline High F1: 0.7461345315184353
Baseline Balanced Accuracy: 0.8118578033366104


In [105]:
threshold_results = []

for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    
    threshold_preds = baseline_preds.copy()
    
    # If probability of High is high enough, assign High
    threshold_preds[y_proba[:, high_index] >= threshold] = focus_class
    
    high_recall = recall_score(
        y_test,
        threshold_preds,
        labels=[focus_class],
        average="macro",
        zero_division=0
    )
    
    high_f1 = f1_score(
        y_test,
        threshold_preds,
        labels=[focus_class],
        average="macro",
        zero_division=0
    )
    
    balanced_acc = balanced_accuracy_score(y_test, threshold_preds)
    
    threshold_results.append({
        "threshold": threshold,
        "High Recall": high_recall,
        "High F1": high_f1,
        "Balanced Accuracy": balanced_acc
    })

threshold_results_df = pd.DataFrame(threshold_results)

display(threshold_results_df)

,threshold,High Recall,High F1,Balanced Accuracy
0,0.10,0.903617,0.532165,0.849444
1,0.15,0.883151,0.608161,0.856242
2,0.20,0.863636,0.664956,0.858095
3,0.25,0.842932,0.703546,0.856478
4,0.30,0.818658,0.732929,0.852423
5,0.35,0.789386,0.751643,0.845712
6,0.40,0.753451,0.760418,0.836160
7,0.50,0.671823,0.746135,0.811858


In [106]:
final_threshold = 0.40

final_preds = baseline_preds.copy()
final_preds[y_proba[:, high_index] >= final_threshold] = focus_class

print("Final Threshold Classification Report:")
print(classification_report(y_test, final_preds))

Final Threshold Classification Report:
              precision    recall  f1-score   support

        High       0.77      0.75      0.76      4202
         Low       0.92      0.89      0.91     73983
      Medium       0.82      0.86      0.84     47815

    accuracy                           0.88    126000
   macro avg       0.84      0.84      0.84    126000
weighted avg       0.88      0.88      0.88    126000



In [107]:
final_cm = confusion_matrix(y_test, final_preds, labels=classes)

final_cm_df = pd.DataFrame(
    final_cm,
    index=[f"Actual {c}" for c in classes],
    columns=[f"Predicted {c}" for c in classes]
)

display(final_cm_df)

,Predicted High,Predicted Low,Predicted Medium
Actual High,3166,22,1014
Actual Low,0,66076,7907
Actual Medium,959,5644,41212


In [108]:
final_recall_high = recall_score(
    y_test,
    final_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

final_f1_high = f1_score(
    y_test,
    final_preds,
    labels=[focus_class],
    average="macro",
    zero_division=0
)

final_balanced_acc = balanced_accuracy_score(y_test, final_preds)

comparison_df = pd.DataFrame({
    "Metric": ["High Recall", "High F1", "Balanced Accuracy"],
    "Baseline": [baseline_recall_high, baseline_f1_high, baseline_balanced_acc],
    "Threshold Model": [final_recall_high, final_f1_high, final_balanced_acc]
})

display(comparison_df)

,Metric,Baseline,Threshold Model
0,High Recall,0.671823,0.753451
1,High F1,0.746135,0.760418
2,Balanced Accuracy,0.811858,0.836160


For this activity, I selected the High irrigation need class because it was the rarest class and one of the most important classes to identify. Since missing a High irrigation case could be more costly, I wanted to focus on improving how well the model performed on that class.

I used High F1 score as my main evaluation metric because it balances precision and recall. Recall matters because I want to catch actual High irrigation cases, but precision also matters because I do not want the model to over-predict High too often.

To improve the Gaussian Naive Bayes model, I used a PowerTransformer on the numeric features and tuned the model’s var_smoothing parameter. I used the PowerTransformer because Gaussian Naive Bayes assumes the numeric features are roughly normally distributed within each class, and the transformer can help reduce skewness in the data. I also tuned var_smoothing to make the model more stable by adjusting how much smoothing is added to the feature variances.

After tuning the model and testing several thresholds, I chose a threshold of 0.40 because it produced the highest High F1 score in my threshold table. Compared to the baseline/default prediction rule, High recall improved from about 0.672 to 0.753, and High F1 improved from about 0.746 to 0.760. Balanced accuracy also improved from about 0.812 to 0.836.

The main tradeoff was that lower thresholds, such as 0.20, gave higher High recall and balanced accuracy, but they lowered the High F1 score. This means the model caught more actual High cases, but likely made more false High predictions. The 0.40 threshold gave the best balance between precision and recall for the High class.

Naive Bayes was useful as a simple baseline model because it was easy to train, produced predicted probabilities, and allowed me to test different thresholds. The PowerTransformer and var_smoothing tuning improved the model, but it still performed worse than my Random Forest and XGBoost models. This makes sense because Naive Bayes makes stronger assumptions about the data, while tree-based models can capture more complex relationships and feature interactions.